In [ ]:
import pandas as pd
import numpy as np

listings = pd.read_csv('data/listings_clean.csv', parse_dates=['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review'])

print(listings.shape)
print(listings['price'].dtype, listings['last_review'].dtypes)

## Lab 6 - Part 1
### Ex 1

In [ ]:
five = pd.Series([10,15,20,25,30])
print(five.mean(), five.median())

six = pd.Series([10,15,20,25,30, 1293])
print(six.mean().round(1), six.median())

print(listings['number_of_reviews'].mean().round(1), 
      listings['number_of_reviews'].median(), 
      listings['number_of_reviews'].std().round(1))

#### five: 20.0 , 20.0 sic : 232.2 , 22.5 real: 35.8 , 9.0
#### median best represent what you really mean *drum* when you say average number_of_reviews.
### Ex 2

In [ ]:
num_reviews = listings['number_of_reviews']
print(num_reviews.describe(), '\n',
      num_reviews.quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))

q1 = num_reviews.quantile(0.25) 
q3 = num_reviews.quantile(0.75) 
iqr = q3 - q1

print('q1',q1,
      'q3',q3,
      'iqr',iqr)

#### describe() count 4574, mean 35.8, std 78.4, min 0, 25% 2, 50% 9, 75% 29, max 1293.
#### Percentiles: 10% = 0, 25% = 2, 50% = 9, 75% = 29, 90% = 94, 99% = 400. q1 = 2, q3 = 29, iqr = 27
#### IQR more resistant to outliers than Std.
#### 10percentile is is 0. 10% of the total listings have no reviews.
### Ex 3

In [ ]:
upper = q3 + 1.5 * iqr 
lower = q1 - 1.5 * iqr
above_upper = listings[num_reviews > upper]
without_above = listings[num_reviews <= upper]

print('upper', upper, 'lower', lower,
      'above upper', above_upper['id'].count(), 
      'percent_above', (above_upper['id'].count() / listings['id'].count()) * 100)

print(listings[['price', 'room_type']].groupby('room_type').agg(n=('price','count')))
print('\nmedian_price_all', listings[['price']].median())
print(above_upper[['price', 'room_type']].groupby('room_type').agg(n=('price','count')))
print('\nmedian_price_above', above_upper[['price']].median())


#### upper 69.5 lower -38.5 . lower outlier not possible since there are many 0 values.
#### 582 , 12.72% above the upper fence.
#### room_type above upper: Entire home/apt 400, Hotel room 0, Private room 105, Shared room 11
#### Median price all: 1791.0 above: 1459.54
#### We can see that listings with many reviews are cheaper.
### Ex 4


In [ ]:
listings['reviews_outlier'] = listings['number_of_reviews'] > upper
print(listings['reviews_outlier'].isin([True]).sum())

for func in ['count', 'mean', 'median', 'std']:
    print(func,
          (listings.loc[~listings['reviews_outlier'], 'price'].agg(func)
          / listings['price'].agg(func)).round(2)
          * 100, ' %')
    
compare = pd.DataFrame({
    'all': listings['number_of_reviews'].agg(['count', 'mean', 'median', 'std']),
    'without' : listings.loc[~listings['reviews_outlier'], 'number_of_reviews']
        .agg(['count', 'mean', 'median', 'std']),
}).round(1)

for func in ['count', 'mean', 'median', 'std']:
    print(func,
          ((compare['without'][func]
          / compare['all'][func] - 1).round(2)
          * 100), ' %')
    
compare

#### 582 listings in review_outliers=True.
#### With all listings: count 4574, mean 35.8, median 9.0, std 78.4. 
#### Without the outliers: count 3992, mean 12.9, median 7.0, std 15.5
#### Changes: mean -64 percent, median -22 percent, std -80 percent (and count -13 percent the 582 listings that are gone).
#### std has the biggest change and median has the smallest. As is to be expected.
### Ex 5

In [ ]:
listings.groupby('room_type')['number_of_reviews'].describe().round(0)

#### Entire home/apt: count 3760, median 8. Hotel room: count 3, median 56. Private room: count 787, median 12. Shared room: count 24, median 82 (mean 114, max 510). 
#### Shared rooms have the highest median among the groups with enough listings.
#### Hotel rooms have a too small sample size to make a judgement.
### Ex 6

In [ ]:
col = [ 'price', 'accommodates', 'bedrooms', 'availability_365', 
        'review_scores_rating', 'minimum_nights', 'days_since_review']
for c in col:
    print('Corr', c, listings['number_of_reviews'].corr(listings[c]).round(2))

print('with outliers', 
    listings["number_of_reviews"].corr(listings["availability_365"]).round(2),
    '\nwithout outliers',
    listings[~listings["reviews_outlier"]]["number_of_reviews"].corr(listings["availability_365"]).round(2))

#### number_of_reviews against: price -0.09, accommodates -0.05, bedrooms -0.14, availability_365 0.18, review_scores_rating -0.04, minimum_nights -0.07, days_since_review -0.19. All are weak.
#### We see that without the outliers the correlation is negligible.